# Prophet Checkpoint Features: Causal Model-Input Feature Generation

### Purpose & Production Design
In the production SimGuard architecture, Prophet forecasts cannot look into the future of a billing cycle. Instead of fitting across full 30-day cycles and broadcasting forecasts backwards onto earlier days (which introduced lookahead leakage), model-input features are generated via **causal checkpoint forecasting**:

1. **Discrete Checkpoints ($C \in \{10, 20, 25\}$)**: For each SIM and billing cycle, Prophet is fit using only rows where `day_in_cycle < C`.
2. **Causal Availability**: A checkpoint fit completes at `fit_end_date = cycle_start + (C - 1)` and becomes available for inference at `available_from_date = cycle_start + C`.
3. **As-Of Daily Attachment**: Daily inference rows attach the latest available checkpoint forecast via `attach_to_daily()`. Days before Day 10 have `prophet_missing_flag = 1` and NaN forecast fields.
4. **Zero Labels**: This pipeline is completely unsupervised. No anomaly labels are used or loaded.

## 1. Setup, Installs, and Imports

We install required libraries, resolve relative paths for Colab and local execution, import from `src.prophet_forecasting`, and initialize a dedicated MLflow experiment (`simguard-prophet-checkpoints`).

In [5]:
# Cell 1: Setup, Installs, and Imports
%pip install -q prophet pandas numpy matplotlib mlflow pyarrow joblib

import os, sys, json, time, warnings, logging
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mlflow

# Suppress verbose logs
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').disabled = True
logging.getLogger('prophet').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

# Relative path resolution for Colab or local run
DATA_DIR = 'data'
ARTIFACT_DIR = 'artifacts'

if not os.path.isdir(DATA_DIR) and os.path.isdir(os.path.join('..', DATA_DIR)):
    os.chdir('..')

os.makedirs(ARTIFACT_DIR, exist_ok=True)

# Ensure project root is in sys.path
if '.' not in sys.path:
    sys.path.insert(0, '.')

# Import from canonical module src.prophet_forecasting
from src.prophet_forecasting import (
    CYCLE_LEN,
    CHECKPOINTS,
    MIN_HISTORY_DAYS,
    TIER_BUNDLE_MB,
    SIM_START,
    forecast_checkpoint,
    fit_many,
    attach_to_daily,
    recommend_tier,
)

# MLflow setup
os.environ.setdefault('MLFLOW_ALLOW_FILE_STORE', 'true')
MLFLOW_DB = 'sqlite:///mlruns.db'
mlflow.set_tracking_uri(MLFLOW_DB)
mlflow.set_experiment('simguard-prophet-checkpoints')

print(f"CWD          : {os.getcwd()}")
print(f"DATA_DIR     : {os.path.abspath(DATA_DIR)}")
print(f"ARTIFACT_DIR : {os.path.abspath(ARTIFACT_DIR)}")
print(f"MLflow URI   : {mlflow.get_tracking_uri()}")
print(f"Experiment   : simguard-prophet-checkpoints")
print(f"Checkpoints  : {CHECKPOINTS} days | Cycle Length: {CYCLE_LEN} days")

Note: you may need to restart the kernel to use updated packages.
CWD          : c:\Users\ADMIN\Desktop\projects\simguard\ml-service
DATA_DIR     : c:\Users\ADMIN\Desktop\projects\simguard\ml-service\data
ARTIFACT_DIR : c:\Users\ADMIN\Desktop\projects\simguard\ml-service\artifacts
MLflow URI   : sqlite:///mlruns.db
Experiment   : simguard-prophet-checkpoints
Checkpoints  : (10, 20, 25) days | Cycle Length: 30 days



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Load Raw Daily Aggregates (No Labels)

We load `synthetic_cdr_daily_agg.csv` only. No labels are loaded. We rebuild fixed 30-day billing cycles (`SIM_START = 2025-01-01`), calendar dates, and tier bundle limits matching `prophet_burn_rate.ipynb`.

In [6]:
# Cell 2: Load Daily Aggregates and Build Calendar
AGG_PATH = os.path.join(DATA_DIR, 'synthetic_cdr_daily_agg.csv')
agg = pd.read_csv(AGG_PATH, parse_dates=['date'])
agg['date'] = pd.to_datetime(agg['date'])

# Verify no label columns exist in the raw input
LABEL_COLS = {'category', 'start_day', 'end_day', 'duration_days', 'is_anomaly'}
assert not any(col in agg.columns for col in LABEL_COLS), "Unexpected label columns found in daily_agg!"

# Add bundle_mb mapping
agg['bundle_mb'] = agg['tier'].map(TIER_BUNDLE_MB)
assert agg['bundle_mb'].isna().sum() == 0, "Unknown tier detected in daily_agg!"

# Rebuild fixed 30-day billing cycle calendar
agg['day_offset'] = (agg['date'] - SIM_START).dt.days
agg['billing_cycle_num'] = (agg['day_offset'] // CYCLE_LEN).astype(int)
agg['day_in_cycle'] = (agg['day_offset'] % CYCLE_LEN).astype(int)
agg['cycle_start_date'] = SIM_START + pd.to_timedelta(agg['billing_cycle_num'] * CYCLE_LEN, unit='D')
agg['cycle_end_date'] = agg['cycle_start_date'] + pd.Timedelta(days=CYCLE_LEN - 1)

print(f"Daily agg shape   : {agg.shape}")
print(f"Date range        : {agg['date'].min().date()} -> {agg['date'].max().date()}")
print(f"Unique SIMs       : {agg['sim_id'].nunique():,}")
print(f"Cycles present    : {sorted(agg['billing_cycle_num'].unique())}")
print(f"Columns           : {list(agg.columns)}")
print(f"No labels loaded  : Confirmed.")

Daily agg shape   : (90000, 26)
Date range        : 2025-01-01 -> 2025-03-31
Unique SIMs       : 1,000
Cycles present    : [np.int64(0), np.int64(1), np.int64(2)]
Columns           : ['sim_id', 'date', 'total_download_mb', 'total_upload_mb', 'data_session_count', 'data_duration_s', 'hotspot_sessions', 'public_apn_sessions', 'unique_towers', 'geo_flag_count', 'peak_mb', 'offpeak_mb', 'total_mb', 'roaming_sessions', 'billed_mb', 'tier', 'voice_minutes_used', 'sms_count', 'voice_minutes_remaining', 'sms_remaining', 'bundle_mb', 'day_offset', 'billing_cycle_num', 'day_in_cycle', 'cycle_start_date', 'cycle_end_date']
No labels loaded  : Confirmed.


## 3. Subsample Toggle & Runtime Estimation

We provide a `SUBSAMPLE_SIMS` toggle (default `50` SIMs for fast, verified execution; set to `None` for the full 1,000 SIM run). Before fitting across all SIMs, we benchmark a single checkpoint fit to compute runtime estimates.

In [7]:
# Cell 3: Subsample Toggle & Runtime Estimation
SUBSAMPLE_SIMS = None  # Default 50 SIMs; set to None for the full 1,000 SIMs run

all_sims = sorted(agg['sim_id'].unique())
if SUBSAMPLE_SIMS is not None:
    selected_sims = all_sims[:SUBSAMPLE_SIMS]
    sample_df = agg[agg['sim_id'].isin(selected_sims)].copy().reset_index(drop=True)
    print(f"Subsampled to {len(selected_sims)} SIMs ({len(sample_df):,} daily rows).")
else:
    selected_sims = all_sims
    sample_df = agg.copy().reset_index(drop=True)
    print(f"Full run selected: {len(selected_sims)} SIMs ({len(sample_df):,} daily rows).")

groups = sample_df.groupby(['sim_id', 'billing_cycle_num'])
num_groups = groups.ngroups
num_cps = len(CHECKPOINTS)
total_fits = num_groups * num_cps

print(f"Total (sim, cycle) groups : {num_groups:,}")
print(f"Checkpoints per group     : {num_cps} ({CHECKPOINTS})")
print(f"Total fits to execute     : {total_fits:,}")

# Benchmark 1 single fit for runtime projection
test_group_key = list(groups.groups.keys())[0]
test_grp = groups.get_group(test_group_key)
test_bundle = float(test_grp['bundle_mb'].iloc[0])

t_bench_start = time.time()
_ = forecast_checkpoint(test_grp, test_bundle, checkpoint_day=CHECKPOINTS[0])
t_single_fit = time.time() - t_bench_start

n_cpus = os.cpu_count() or 4
est_serial_s = total_fits * t_single_fit
est_parallel_s = est_serial_s / max(1, n_cpus * 0.75)  # conservative parallel scaling factor

print(f"\n--- Runtime Estimation ---")
print(f"Single fit benchmark time : {t_single_fit:.3f} s")
print(f"Available CPU cores       : {n_cpus}")
print(f"Projected serial time     : {est_serial_s:.1f} s ({est_serial_s/60:.2f} min)")
print(f"Projected parallel time   : ~{est_parallel_s:.1f} s ({est_parallel_s/60:.2f} min)")

Full run selected: 1000 SIMs (90,000 daily rows).
Total (sim, cycle) groups : 3,000
Checkpoints per group     : 3 ((10, 20, 25))
Total fits to execute     : 9,000

--- Runtime Estimation ---
Single fit benchmark time : 2.694 s
Available CPU cores       : 12
Projected serial time     : 24245.0 s (404.08 min)
Projected parallel time   : ~2693.9 s (44.90 min)


## 4. Run `fit_many`

We run `fit_many` with joblib parallelism (`n_jobs=-1`). We log runtime and save any fit errors to `artifacts/prophet_checkpoint_errors.csv`.

In [8]:
# Cell 4: Run fit_many
print(f"Starting fit_many for {num_groups} groups across checkpoints {CHECKPOINTS}...")
t0 = time.time()

fc_df, errors = fit_many(groups, checkpoints=CHECKPOINTS, n_jobs=-1)
runtime_seconds = time.time() - t0

print(f"\nfit_many execution completed:")
print(f"  Runtime        : {runtime_seconds:.2f} s ({runtime_seconds/60:.2f} min)")
print(f"  Throughput     : {len(fc_df) / max(0.001, runtime_seconds):.1f} fits/sec")
print(f"  Forecast rows  : {len(fc_df):,}")
print(f"  Error count    : {len(errors):,}")

# Save errors if any
err_path = os.path.join(ARTIFACT_DIR, 'prophet_checkpoint_errors.csv')
if len(errors) > 0:
    err_df = pd.DataFrame(errors)
    err_df.to_csv(err_path, index=False)
    print(f"  [WARNING] Saved {len(errors)} error records to: {err_path}")
else:
    print(f"  [OK] Zero fit errors encountered.")
    if os.path.exists(err_path):
        os.remove(err_path)

Starting fit_many for 3000 groups across checkpoints (10, 20, 25)...

fit_many execution completed:
  Runtime        : 2146.64 s (35.78 min)
  Throughput     : 4.2 fits/sec
  Forecast rows  : 9,000
  Error count    : 0
  [OK] Zero fit errors encountered.


## 5. Save Checkpoint Forecasts Table

We save the fitted checkpoint forecasts to `data/prophet_checkpoints.parquet` (one row per SIM, cycle, and checkpoint).

In [9]:
# Cell 5: Save Checkpoint Forecasts Parquet
CP_OUT_PATH = os.path.join(DATA_DIR, 'prophet_checkpoints.parquet')
fc_df.to_parquet(CP_OUT_PATH, index=False)

print(f"Saved checkpoint forecasts to: {CP_OUT_PATH}")
print(f"File shape : {fc_df.shape}")
print(f"Columns    : {list(fc_df.columns)}")
print("\nSample forecast rows:")
print(fc_df.head(6)[['sim_id', 'billing_cycle_num', 'checkpoint_day', 'fit_end_date', 'available_from_date', 'predicted_final_cum', 'projected_overage_mb', 'recommendation']])

Saved checkpoint forecasts to: data\prophet_checkpoints.parquet
File shape : (9000, 10)
Columns    : ['sim_id', 'billing_cycle_num', 'checkpoint_day', 'fit_end_date', 'available_from_date', 'predicted_final_cum', 'projected_overage_mb', 'days_to_depletion', 'recommendation', 'recommended_tier']

Sample forecast rows:
   sim_id  billing_cycle_num  checkpoint_day fit_end_date available_from_date  \
0       0                  0              10   2025-01-10          2025-01-11   
1       0                  0              20   2025-01-20          2025-01-21   
2       0                  0              25   2025-01-25          2025-01-26   
3       0                  1              10   2025-02-09          2025-02-10   
4       0                  1              20   2025-02-19          2025-02-20   
5       0                  1              25   2025-02-24          2025-02-25   

   predicted_final_cum  projected_overage_mb recommendation  
0         10961.722697         -29998.277303      d

## 6. Causal Daily Attachment & Sanity Checks

We attach the latest available checkpoint forecast to every daily row via `attach_to_daily()`, save to `data/prophet_checkpoint_features.parquet`, and execute all 5 required sanity checks.

In [10]:
# Cell 6: Attach to Daily Grid and Run Sanity Checks
daily_with_fc = attach_to_daily(sample_df, fc_df)

SAVE_COLS = [
    'sim_id',
    'date',
    'billing_cycle_num',
    'projected_overage_mb',
    'days_to_depletion',
    'recommendation',
    'prophet_missing_flag',
    'fit_end_date',
]
daily_feat = daily_with_fc[SAVE_COLS].copy()

FEAT_OUT_PATH = os.path.join(DATA_DIR, 'prophet_checkpoint_features.parquet')
daily_feat.to_parquet(FEAT_OUT_PATH, index=False)
print(f"Saved daily features to: {FEAT_OUT_PATH}")
print(f"Feature table shape: {daily_feat.shape}")

# -------------------------------------------------------------
# SANITY CHECKS
# -------------------------------------------------------------
print("\n=== RUNNING SANITY CHECKS ===")

# Check 1: fit_end_date < date for all rows with forecast
has_fc = daily_feat['prophet_missing_flag'] == 0
violations = daily_feat.loc[has_fc, 'fit_end_date'] >= daily_feat.loc[has_fc, 'date']
assert violations.sum() == 0, f"Check 1 Failed: {violations.sum()} causality violations!"
print(f"[PASS] Check 1: fit_end_date < date across all {has_fc.sum():,} rows with forecast (0 violations).")

# Check 2: Row count equals SIMs x 90 days
expected_rows = len(selected_sims) * 90
assert len(daily_feat) == expected_rows, f"Check 2 Failed: expected {expected_rows} rows, got {len(daily_feat)}!"
print(f"[PASS] Check 2: Row count ({len(daily_feat):,}) equals {len(selected_sims)} SIMs x 90 days.")

# Check 3: Share of rows with prophet_missing_flag = 1 is about 10/30 (~33.3%)
missing_share = (daily_feat['prophet_missing_flag'] == 1).mean()
print(f"[PASS] Check 3: prophet_missing_flag = 1 share is {missing_share:.4f} (expected ~10/30 = {10/30:.4f}).")
assert 0.30 <= missing_share <= 0.36, f"Missing flag share {missing_share:.4f} outside expected range!"

# Check 4: No label columns in saved files
LABEL_COLS = {'category', 'start_day', 'end_day', 'duration_days', 'is_anomaly', 'label'}
assert not any(col in daily_feat.columns for col in LABEL_COLS), "Check 4 Failed: Label columns found in daily features!"
assert not any(col in fc_df.columns for col in LABEL_COLS), "Check 4 Failed: Label columns found in checkpoint forecasts!"
print(f"[PASS] Check 4: Strictly zero label columns in saved feature parquet files.")

# Check 5: No NaN anywhere except forecast fields on missing-flag rows
non_forecast_cols = ['sim_id', 'date', 'billing_cycle_num', 'prophet_missing_flag']
assert daily_feat[non_forecast_cols].isna().sum().sum() == 0, "Check 5 Failed: NaNs found in non-forecast metadata columns!"

# On non-missing rows, projected_overage_mb, recommendation, fit_end_date must NOT be NaN
complete_forecast_cols = ['projected_overage_mb', 'recommendation', 'fit_end_date']
assert daily_feat.loc[has_fc, complete_forecast_cols].isna().sum().sum() == 0, "Check 5 Failed: NaNs found in forecast cols on valid rows!"
print(f"[PASS] Check 5: No NaNs anywhere except forecast fields on missing-flag rows (and days_to_depletion when non-depleting).")

print("\n=== ALL SANITY CHECKS PASSED ===")

Saved daily features to: data\prophet_checkpoint_features.parquet
Feature table shape: (90000, 8)

=== RUNNING SANITY CHECKS ===
[PASS] Check 1: fit_end_date < date across all 60,000 rows with forecast (0 violations).
[PASS] Check 2: Row count (90,000) equals 1000 SIMs x 90 days.
[PASS] Check 3: prophet_missing_flag = 1 share is 0.3333 (expected ~10/30 = 0.3333).
[PASS] Check 4: Strictly zero label columns in saved feature parquet files.
[PASS] Check 5: No NaNs anywhere except forecast fields on missing-flag rows (and days_to_depletion when non-depleting).

=== ALL SANITY CHECKS PASSED ===


## 7. Forecast Quality Evaluation (Label-Free MAE)

We evaluate how forecast accuracy improves as the cycle progresses without touching any anomaly labels. We compute the Mean Absolute Error (MAE) between `predicted_final_cum` and the actual cumulative total from `daily_agg` for each checkpoint day ($C \in \{10, 20, 25\}$). We plot the MAE progression and save to `artifacts/checkpoint_mae.png`.

In [11]:
# Cell 7: Forecast Quality Evaluation and MAE Plot
actual_totals = (
    sample_df.groupby(['sim_id', 'billing_cycle_num'])['total_mb']
    .sum()
    .rename('actual_cycle_total_mb')
    .reset_index()
)

eval_df = fc_df.merge(actual_totals, on=['sim_id', 'billing_cycle_num'])
eval_df['abs_error_mb'] = (eval_df['predicted_final_cum'] - eval_df['actual_cycle_total_mb']).abs()

mae_by_cp = eval_df.groupby('checkpoint_day')['abs_error_mb'].mean()

print("--- Forecast Quality (Mean Absolute Error vs Actual Cycle End Total) ---")
for cp, mae in mae_by_cp.items():
    print(f"  Checkpoint Day {cp:>2}: MAE = {mae:>9.2f} MB ({mae/1024:>6.2f} GB)")

# Plot MAE progression
fig, ax = plt.subplots(figsize=(8, 5))
cps = list(mae_by_cp.index)
maes = list(mae_by_cp.values)

ax.plot(cps, maes, marker='o', linewidth=2.5, markersize=8, color='#1f77b4', label='Mean Absolute Error')
for x, y in zip(cps, maes):
    ax.annotate(
        f"{y:,.1f} MB\n({y/1024:.2f} GB)",
        (x, y),
        textcoords="offset points",
        xytext=(0, 12),
        ha='center',
        fontsize=10,
        fontweight='bold',
    )

ax.set_title('Prophet Forecast Accuracy Progression Across Checkpoints', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Checkpoint Day in Billing Cycle', fontsize=11)
ax.set_ylabel('Mean Absolute Error (MB)', fontsize=11)
ax.set_xticks(cps)
ax.set_xticklabels([f"Day {cp}\n({cp} days observed)" for cp in cps])
ax.grid(True, linestyle='--', alpha=0.6)
ax.set_ylim(0, max(maes) * 1.35)
plt.tight_layout()

mae_plot_path = os.path.join(ARTIFACT_DIR, 'checkpoint_mae.png')
fig.savefig(mae_plot_path, dpi=150)
plt.close(fig)
print(f"\nSaved MAE plot to: {mae_plot_path}")

--- Forecast Quality (Mean Absolute Error vs Actual Cycle End Total) ---
  Checkpoint Day 10: MAE =   2436.02 MB (  2.38 GB)
  Checkpoint Day 20: MAE =    716.31 MB (  0.70 GB)
  Checkpoint Day 25: MAE =    346.67 MB (  0.34 GB)

Saved MAE plot to: artifacts\checkpoint_mae.png
